In [6]:
import pandas as pd
from pytube import Playlist, YouTube
from youtube_transcript_api import YouTubeTranscriptApi
import time

# 1. Sources avec indication de la langue dominante attendue
sources_a_analyser = [
    {"type": "playlist", "url": "https://www.youtube.com/playlist?list=PL0YDwQCRiJMVLR63JUiVtAKYJZJK-c0ri", "label": "The Day We Spend_CaminoFrancès", "pays": "USA"},
    {"type": "playlist", "url": "https://www.youtube.com/playlist?list=PL0YDwQCRiJMU9ee71FPN27CasJFOofUcX", "label": "The Day We Spend_ViaPodiensis", "pays": "USA"},
    {"type": "playlist", "url": "https://www.youtube.com/playlist?list=PL0YDwQCRiJMVK_UL-BpJPLowhtGt-UvLD", "label": "The Day We Spend_Camino_Primitivo", "pays": "USA"},
    {"type": "playlist", "url": "https://www.youtube.com/playlist?list=PL0YDwQCRiJMVY9cODWmNq9cnjEOiA9grd", "label": "The Day we Spend_CaminoInvierno", "pays": "USA"},
    {"type": "playlist", "url": "https://www.youtube.com/playlist?list=PL0YDwQCRiJMVCegVyHjrO7_WC3GS_Q-9v", "label": "The Day We spend_CaminoDelNorte", "pays": "USA"},
    {"type": "playlist", "url": "https://www.youtube.com/watch?v=yA2Vq4de2SU&list=PLrsSeUdeL-caz3C06FodPhURKBu5MJ9JD", "label": "Sisters 2 Santiago_GR65", "pays": "USA"},
    {"type": "playlist", "url": "https://www.youtube.com/watch?v=BGfMOB75UZ0&list=PLrsSeUdeL-cZwMzvA94EZCy3CW2gMQeCj", "label": "Sisters 2 Santiago_CaminoFrancès", "pays": "USA"},
    {"type": "playlist", "url": "https://www.youtube.com/watch?v=n57XMEpQNZA&list=PLrsSeUdeL-ca51g4N_zWb72rluOGBV_c1", "label": "Sisters 2 Santiago_CaminoPortuguese", "pays": "USA"},
    {"type": "playlist", "url": "https://www.youtube.com/playlist?list=PLFQmjLhzwpCntIzkqzVbid_1BgPpXTx-k", "label": "Des Gites et des Gens - Voie du Puy-en-Velay", "pays": "FR"},
    {"type": "playlist", "url": "https://www.youtube.com/watch?v=rn8SSetb-7g&list=PL0GGKbG49dlxr9OFckCQOddpQMAL8LWwO", "label": "Vlogs TravelMike", "pays": "FR"},
    {"type": "playlist", "url": "https://www.youtube.com/watch?v=tO4qTSKuvuw&list=PLTQH-UfP0HiqFbzP3e-ycC6sLsYWT-BsZ", "label": "Vlogs Tof Aventurier St Jacques", "pays": "FR"},
    {"type": "playlist", "url": "https://www.youtube.com/watch?v=zryKRSBG49s&list=PLTQH-UfP0HipYrQglnZSKI-dGF0afQAbA", "label": "Vlogs Tof Aventurier variante Rocamadour", "pays": "FR"},
    {"type": "playlist", "url": "https://www.youtube.com/watch?v=iiLWeZTo2nk&list=PLTQH-UfP0HiradtwNXjrmC0XE86UnKki4", "label": "Vlogs Tof Aventurier Via de la Plata", "pays": "FR"},
    {"type": "playlist", "url": "https://www.youtube.com/watch?v=KJvDzxOya94&list=PLTQH-UfPe0Hiq_smXbWDKZb-zUUw52V-R", "label": "Vlogs Tof Aventurier Voie d'Arles", "pays": "FR"},
    {"type": "playlist", "url": "https://www.youtube.com/watch?v=jbOaawyrSZo", "label": "Vlog Pelerin", "pays": "USA"},
    {"type": "video", "url": "https://www.youtube.com/watch?v=7NqgAzEoKl8", "label": "David_Wen_CaminoFrancès", "pays": "USA"},
    {"type": "video", "url": "https://www.youtube.com/watch?v=grQZapa8yYE", "label": "David_Wen_CaminoPortuguese", "pays": "USA"},
    {"type": "video", "url": "https://www.youtube.com/watch?v=9PNwL4qunnU", "label": "David_Wen_CaminoTheEnglishWay", "pays": "USA"},
    {"type": "video", "url": "https://www.youtube.com/watch?v=pEFFrG_6o5E&list=PLMbNA0H3vQg5ICsg-nKa3EyqdcV5nmNTY", "label": "Vlog PhilippeProhom", "pays": "FR"},
    {"type": "video", "url": "https://www.youtube.com/watch?v=VPT0lJKidJo", "label": "VideLog", "pays": "FR"},
    
]

def get_transcript_with_lang(video_id):
    try:
        transcript_list = YouTubeTranscriptApi.list_transcripts(video_id)
        
        # On essaie de trouver une langue manuelle d'abord, sinon on prend l'auto
        # On élargit la liste des langues pour être sûr de capter quelque chose
        try:
            transcript = transcript_list.find_transcript(['fr', 'en', 'es', 'ko', 'de'])
        except:
            # Si aucune des langues ci-dessus n'est trouvée, on prend la première dispo
            transcript = transcript_list.find_generated_transcript(['fr', 'en', 'es', 'ko', 'de'])
            
        data = transcript.fetch()
        full_text = " ".join([t['text'] for t in data])
        return full_text, transcript.language_code
    except Exception as e:
        # Utile pour débugger : décommentez la ligne suivante si besoin
        # print(f"Erreur transcript pour {video_id}: {e}")
        return None, None

def process_international_sources(sources):
    all_records = []
    
    for source in sources:
        print(f"\n--- Analyse de : {source['label']} ---")
        
        urls_a_traiter = []
        try:
            if source['type'] == 'playlist':
                pl = Playlist(source['url'])
                # On force le chargement des URLs
                urls_a_traiter = list(pl.video_urls)
            else:
                urls_a_traiter = [source['url']]
        except Exception as e:
            print(f"Erreur accès source : {e}")
            continue

        for url in urls_a_traiter:
            try:
                video = YouTube(url)
                video_id = video.video_id
                
                # On récupère le texte
                text, lang = get_transcript_with_lang(video_id)
                
                if text:
                    all_records.append({
                        'pays_cible': source.get('pays', 'Inconnu'),
                        'langue_detectee': lang,
                        'source_label': source['label'],
                        'nom_chaine': video.author,
                        'titre': video.title,
                        'date_publi': video.publish_date,
                        'transcript': text,
                        'url': url
                    })
                    print(f"  ✓ Récupéré : {video.title[:40]}")
                else:
                    print(f"  ✗ Aucun transcript trouvé pour : {video.title[:40]}")
            except Exception as e:
                print(f"  ✗ Erreur vidéo {url}: {e}")
            
            time.sleep(1) # Un peu plus de repos pour éviter le blocage

    return pd.DataFrame(all_records)

In [9]:
import pandas as pd
from youtube_transcript_api import YouTubeTranscriptApi
import yt_dlp
import time

# 1. Votre dictionnaire de sources
sources_a_analyser = [
    {"type": "playlist", "url": "https://www.youtube.com/playlist?list=PL0YDwQCRiJMVLR63JUiVtAKYJZJK-c0ri", "label": "The Day We Spend_CaminoFrancès", "pays": "USA"},
    {"type": "playlist", "url": "https://www.youtube.com/playlist?list=PL0YDwQCRiJMU9ee71FPN27CasJFOofUcX", "label": "The Day We Spend_ViaPodiensis", "pays": "USA"},
    {"type": "playlist", "url": "https://www.youtube.com/playlist?list=PL0YDwQCRiJMVK_UL-BpJPLowhtGt-UvLD", "label": "The Day We Spend_Camino_Primitivo", "pays": "USA"},
    {"type": "playlist", "url": "https://www.youtube.com/playlist?list=PL0YDwQCRiJMVY9cODWmNq9cnjEOiA9grd", "label": "The Day we Spend_CaminoInvierno", "pays": "USA"},
    {"type": "playlist", "url": "https://www.youtube.com/playlist?list=PL0YDwQCRiJMVCegVyHjrO7_WC3GS_Q-9v", "label": "The Day We spend_CaminoDelNorte", "pays": "USA"},
    {"type": "playlist", "url": "https://www.youtube.com/watch?v=yA2Vq4de2SU&list=PLrsSeUdeL-caz3C06FodPhURKBu5MJ9JD", "label": "Sisters 2 Santiago_GR65", "pays": "USA"},
    {"type": "playlist", "url": "https://www.youtube.com/watch?v=BGfMOB75UZ0&list=PLrsSeUdeL-cZwMzvA94EZCy3CW2gMQeCj", "label": "Sisters 2 Santiago_CaminoFrancès", "pays": "USA"},
    {"type": "playlist", "url": "https://www.youtube.com/watch?v=n57XMEpQNZA&list=PLrsSeUdeL-ca51g4N_zWb72rluOGBV_c1", "label": "Sisters 2 Santiago_CaminoPortuguese", "pays": "USA"},
    {"type": "playlist", "url": "https://www.youtube.com/playlist?list=PLFQmjLhzwpCntIzkqzVbid_1BgPpXTx-k", "label": "Des Gites et des Gens - Voie du Puy-en-Velay", "pays": "FR"},
    {"type": "playlist", "url": "https://www.youtube.com/watch?v=rn8SSetb-7g&list=PL0GGKbG49dlxr9OFckCQOddpQMAL8LWwO", "label": "Vlogs TravelMike", "pays": "FR"},
    {"type": "playlist", "url": "https://www.youtube.com/watch?v=tO4qTSKuvuw&list=PLTQH-UfP0HiqFbzP3e-ycC6sLsYWT-BsZ", "label": "Vlogs Tof Aventurier St Jacques", "pays": "FR"},
    {"type": "playlist", "url": "https://www.youtube.com/watch?v=zryKRSBG49s&list=PLTQH-UfP0HipYrQglnZSKI-dGF0afQAbA", "label": "Vlogs Tof Aventurier variante Rocamadour", "pays": "FR"},
    {"type": "playlist", "url": "https://www.youtube.com/watch?v=iiLWeZTo2nk&list=PLTQH-UfP0HiradtwNXjrmC0XE86UnKki4", "label": "Vlogs Tof Aventurier Via de la Plata", "pays": "FR"},
    {"type": "playlist", "url": "https://www.youtube.com/watch?v=KJvDzxOya94&list=PLTQH-UfPe0Hiq_smXbWDKZb-zUUw52V-R", "label": "Vlogs Tof Aventurier Voie d'Arles", "pays": "FR"},
    {"type": "playlist", "url": "https://www.youtube.com/watch?v=jbOaawyrSZo", "label": "Vlog Pelerin", "pays": "USA"},
    {"type": "video", "url": "https://www.youtube.com/watch?v=7NqgAzEoKl8", "label": "David_Wen_CaminoFrancès", "pays": "USA"},
    {"type": "video", "url": "https://www.youtube.com/watch?v=grQZapa8yYE", "label": "David_Wen_CaminoPortuguese", "pays": "USA"},
    {"type": "video", "url": "https://www.youtube.com/watch?v=9PNwL4qunnU", "label": "David_Wen_CaminoTheEnglishWay", "pays": "USA"},
    {"type": "video", "url": "https://www.youtube.com/watch?v=pEFFrG_6o5E&list=PLMbNA0H3vQg5ICsg-nKa3EyqdcV5nmNTY", "label": "Vlog PhilippeProhom", "pays": "FR"},
    {"type": "video", "url": "https://www.youtube.com/watch?v=VPT0lJKidJo", "label": "VideLog", "pays": "FR"},
]

def get_video_info_yt_dlp(url):
    """Utilise yt-dlp pour extraire les infos de playlist ou vidéo sans erreur 400"""
    ydl_opts = {'quiet': True, 'extract_flat': True, 'force_generic_extract': True}
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        try:
            info = ydl.extract_info(url, download=False)
            if 'entries' in info: # C'est une playlist
                return info['entries'], True
            else: # C'est une vidéo seule
                return [info], False
        except Exception as e:
            print(f"Erreur d'accès à l'URL {url}: {e}")
            return [], False

def get_transcript(video_id):
    """Récupère n'importe quel transcript disponible (Manuel ou Auto)"""
    try:
        transcript_list = YouTubeTranscriptApi.list_transcripts(video_id)
        
        # Stratégie : On essaie de récupérer n'importe quelle langue 
        # parmi notre liste, qu'elle soit manuelle ou générée par l'IA
        try:
            # On tente d'abord de trouver une version manuelle ou auto dans nos langues cibles
            transcript = transcript_list.find_transcript(['fr', 'en', 'es', 'ko'])
        except:
            # Si échec, on force la récupération de n'importe quel transcript généré automatiquement
            # C'est souvent ici que se cachent les données des Vlogs
            transcript = transcript_list.find_generated_transcript(['fr', 'en', 'es', 'ko'])
            
        data = transcript.fetch()
        full_text = " ".join([t['text'] for t in data])
        return full_text, transcript.language_code
    
    except Exception as e:
        # Si vraiment rien n'est trouvé (vidéo musicale ou sans parole)
        return None, None

def run_extraction(sources):
    all_data = []
    
    for source in sources:
        print(f"\n--- {source['label']} ({source['pays']}) ---")
        items, is_playlist = get_video_info_yt_dlp(source['url'])
        
        for item in items:
            v_id = item.get('id') or item.get('url')
            v_title = item.get('title', 'Titre inconnu')
            
            # Extraction du transcript
            text, lang = get_transcript(v_id)
            
            if text:
                all_data.append({
                    'source_label': source['label'],
                    'pays_origine': source['pays'],
                    'titre_video': v_title,
                    'transcript': text,
                    'langue': lang,
                    'video_url': f"https://www.youtube.com/watch?v={v_id}"
                })
                print(f"  ✓ {v_title[:40]}... [{lang}]")
            else:
                print(f"  ✗ Pas de texte pour {v_title[:40]}...")
            
            time.sleep(0.5) # Sécurité
            
    return pd.DataFrame(all_data)

# EXÉCUTION
df_pelerins = run_extraction(sources_a_analyser)

# SAUVEGARDE
if not df_pelerins.empty:
    df_pelerins.to_csv("donnees_pelerins_compostelle.csv", index=False, encoding='utf-8-sig')
    print(f"\nTerminé ! {len(df_pelerins)} témoignages extraits.")


--- The Day We Spend_CaminoFrancès (USA) ---
  ✗ Pas de texte pour Camino Frances: St-Jean-Pied-de-Port to ...
  ✗ Pas de texte pour Camino Frances: Pamplona to Navarette (C...
  ✗ Pas de texte pour Camino Frances: Navarette to Burgos (Cam...
  ✗ Pas de texte pour Camino Frances: Burgos to León (Camino d...
  ✗ Pas de texte pour Camino Frances: León to Samos (Camino de...
  ✗ Pas de texte pour Camino Frances: Samos to Santiago de Com...
  ✗ Pas de texte pour How Much Does the Camino de Santiago Cos...

--- The Day We Spend_ViaPodiensis (USA) ---
  ✗ Pas de texte pour Le Puy Camino: Le Puy-en-Velay to Saint-...
  ✗ Pas de texte pour Le Puy Camino: Saint-Alban to Saint-Côme...
  ✗ Pas de texte pour Le Puy Camino: St-Côme d'Olt to Conques ...
  ✗ Pas de texte pour Le Puy Camino: Conques to Cahors | Part ...
  ✗ Pas de texte pour Le Puy Camino: Cahors to Lectoure | Part...
  ✗ Pas de texte pour Le Puy Camino: Lectoure to Arzacq | Part...
  ✗ Pas de texte pour Le Puy Camino: Arzacq to St-J

  ✗ Pas de texte pour Titre inconnu...

--- Vlog Pelerin (USA) ---


  ✗ Pas de texte pour Camino de Santiago: A LIFE-CHANGING ADVE...

--- David_Wen_CaminoFrancès (USA) ---


  ✗ Pas de texte pour Camino de Santiago - Camino Frances | 50...

--- David_Wen_CaminoPortuguese (USA) ---


  ✗ Pas de texte pour Camino de Santiago - Camino Portugues 20...

--- David_Wen_CaminoTheEnglishWay (USA) ---


  ✗ Pas de texte pour Camino de Santiago: The English Way (Ing...

--- Vlog PhilippeProhom (FR) ---
  ✗ Pas de texte pour Ce que je vis quand je ne chante pas (de...

--- VideLog (FR) ---


  ✗ Pas de texte pour Chemin de COMPOSTELLE complet...


Test ASR

In [11]:
import yt_dlp
import whisper
import pandas as pd
import os
import time

# 1. Configuration du modèle Whisper
# 'base' est idéal pour les tests : très rapide et déjà très performant.
print("Chargement de l'intelligence artificielle...")
model = whisper.load_model("base")

# 2. Définition de la Playlist Test
# J'ai sélectionné une playlist courte de témoignages sur le Chemin
sources_test = [
    {
        "type": "playlist", 
        "url": "https://www.youtube.com/playlist?list=PLFQmjLhzwpCntIzkqzVbid_1BgPpXTx-k", 
        "label": "TEST_VOIE_DU_PUY", 
        "pays": "FR"
    }
]

def process_with_whisper(sources_list, limit_videos=3):
    results = []
    
    # Options de téléchargement optimisées pour l'audio
    ydl_opts = {
        'format': 'bestaudio/best',
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'mp3',
            'preferredquality': '128',
        }],
        'outtmpl': 'temp_audio.%(ext)s',
        'quiet': True,
        'noplaylist': True
    }

    for source in sources_list:
        print(f"\nSource : {source['label']}")
        
        # Extraction des entrées de la playlist
        with yt_dlp.YoutubeDL({'extract_flat': True, 'quiet': True}) as ydl:
            info = ydl.extract_info(source['url'], download=False)
            video_entries = info.get('entries', [info])[:limit_videos] # On limite aux 3 premières pour le test

        for entry in video_entries:
            video_url = entry.get('url') if 'http' in entry.get('url', '') else f"https://www.youtube.com/watch?v={entry['id']}"
            video_title = entry.get('title', 'Sans titre')
            
            try:
                print(f"  -> Analyse de : {video_title[:50]}...")
                
                # Téléchargement
                with yt_dlp.YoutubeDL(ydl_opts) as ydl:
                    ydl.download([video_url])
                
                # Transcription
                audio_file = "temp_audio.mp3"
                # On force la langue 'fr' si on sait que c'est du français pour gagner en précision
                transcription = model.transcribe(audio_file, verbose=False)
                
                results.append({
                    'titre': video_title,
                    'transcript': transcription['text'],
                    'langue': transcription['language'],
                    'label': source['label']
                })
                
                if os.path.exists(audio_file):
                    os.remove(audio_file)
                print(f"     ✅ Transcription réussie ({len(transcription['text'].split())} mots)")

            except Exception as e:
                print(f"     ❌ Erreur : {e}")
            
    return pd.DataFrame(results)

# 3. Lancement du test
df_test = process_with_whisper(sources_test, limit_videos=2) # On teste sur les 2 premières vidéos

# 4. Affichage du résultat qualitatif
print("\n--- APERÇU DES DONNÉES EXTRAITES ---")
if not df_test.empty:
    for i, row in df_test.iterrows():
        print(f"\nVidéo : {row['titre']}")
        print(f"Extrait : {row['transcript'][:300]}...") 
else:
    print("Le DataFrame est vide. Vérifiez l'installation de FFmpeg.")

Chargement de l'intelligence artificielle...


100%|███████████████████████████████████████| 139M/139M [00:03<00:00, 39.6MiB/s]



Source : TEST_VOIE_DU_PUY
  -> Analyse de : Des Gîtes et des Gens - Ep. Zero - Le Grand départ...


ERROR: Postprocessing: ffprobe and ffmpeg not found. Please install or provide the path using --ffmpeg-location


     ❌ Erreur : ERROR: Postprocessing: ffprobe and ffmpeg not found. Please install or provide the path using --ffmpeg-location
  -> Analyse de : Gîtes and People - Ep 01 - Le Puy en Velay - Grand...


ERROR: Postprocessing: ffprobe and ffmpeg not found. Please install or provide the path using --ffmpeg-location


     ❌ Erreur : ERROR: Postprocessing: ffprobe and ffmpeg not found. Please install or provide the path using --ffmpeg-location

--- APERÇU DES DONNÉES EXTRAITES ---
Le DataFrame est vide. Vérifiez l'installation de FFmpeg.
